# 02 — LoRA Rank Sweep on Banking77

Trains LoRA adapters at `r = 4, 8, 16, 32` on the same task and split as notebook 01, to find where
extra rank stops buying accuracy.

Runs independently of notebook 01 — it loads its own data and trains its own models. If 01's results
are present in `results/`, its full-fine-tune accuracy is picked up automatically as a reference line
on the plot; if not, the plot simply omits that line.

**Runtime:** GPU required. Four models with early stopping — budget 1.5–2 hours.

## Setup

In [ ]:
!pip install -q transformers "datasets<4.0" peft accelerate evaluate torch scikit-learn
# Kaggle's base image ships torchao 0.10, which peft version-checks and rejects when injecting
# LoRA adapters. This project never uses torchao, so removing it sidesteps the check.
!pip uninstall -y -q torchao

In [ ]:
import gc
import json
import os
import time

import evaluate
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

BATCH_SIZE = 16
COLS = ["input_ids", "attention_mask", "labels"]

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print("WARNING: no GPU detected — this notebook trains 4 models; CPU is impractical.")

# Repo layout puts results/ next to notebooks/; hosted runtimes (Kaggle, Colab) have no
# parent dir to write to, so fall back to a local results/ and download it from the output panel.
RESULTS_DIR = "../results" if os.path.isdir("../results") else "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Writing results to:", os.path.abspath(RESULTS_DIR))

# Reference line for the accuracy plot, read from notebook 01's output if it is available.
FULL_FINETUNE_ACCURACY = None
_baseline_path = os.path.join(RESULTS_DIR, "01_baseline_comparison.json")
if os.path.exists(_baseline_path):
    with open(_baseline_path) as fh:
        for row in json.load(fh):
            if row["method"] == "full_finetune":
                FULL_FINETUNE_ACCURACY = row["accuracy"]
print("Full fine-tune reference accuracy:", FULL_FINETUNE_ACCURACY)

## 1. Data

In [ ]:
dataset = load_dataset("PolyAI/banking77", trust_remote_code=True)
label_names = dataset["train"].features["label"].names
num_labels = len(label_names)
print(f"Classes: {num_labels}")

In [ ]:
MODEL_NAME = "bert-base-uncased"
SEED = 42
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=64)


# padding="max_length" makes every batch fixed-size, so Trainer needs no data collator --
# which also sidesteps the tokenizer= -> processing_class= rename across transformers versions.
tokenized = dataset.map(tokenize_fn, batched=True)
tokenized = tokenized.rename_column("label", "labels")

# Banking77 ships train/test only. Early stopping and best-checkpoint selection have to run
# against data the final number is NOT reported on, so carve a stratified validation split
# out of train and leave test untouched until the final evaluate().
split = tokenized["train"].train_test_split(
    test_size=0.1, seed=SEED, stratify_by_column="labels"
)
train_ds = split["train"].with_format("torch", columns=COLS)
val_ds = split["test"].with_format("torch", columns=COLS)
test_ds = tokenized["test"].with_format("torch", columns=COLS)

print(f"Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}")
n_gpu = max(torch.cuda.device_count(), 1)
print(f"GPUs visible: {torch.cuda.device_count()} -> effective train batch = {BATCH_SIZE * n_gpu}")

## 2. Shared helpers

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "macro_f1": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"],
    }


def count_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total


def reset_peak_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        for i in range(torch.cuda.device_count()):
            torch.cuda.reset_peak_memory_stats(i)


def peak_memory_mb():
    """Highest per-GPU allocation. Kaggle's T4 x2 runs DataParallel, so device 0 alone
    under-reports; the max across devices is what a single card would need to hold."""
    if not torch.cuda.is_available():
        return None
    return max(
        torch.cuda.max_memory_allocated(i) for i in range(torch.cuda.device_count())
    ) / (1024 ** 2)


def epoch_history(trainer):
    """Per-epoch eval metrics pulled out of Trainer's log history."""
    return [
        {
            "epoch": round(rec["epoch"], 2),
            "eval_loss": rec.get("eval_loss"),
            "accuracy": rec.get("eval_accuracy"),
            "macro_f1": rec.get("eval_macro_f1"),
        }
        for rec in trainer.state.log_history
        if "eval_accuracy" in rec
    ]


def release_cuda():
    """Call immediately after `del`-ing the model/trainer names in the calling scope.
    The `del` has to happen at the call site: passing objects into a helper and deleting
    them there only drops the helper's own reference, so nothing is actually freed."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 3. Sweep

`alpha` is held at `2r` so the `alpha/r` scaling stays constant across ranks — otherwise a change in
accuracy could be the scaling moving rather than the rank. Each rank trains with the same early
stopping protocol as notebook 01, so ranks that need more epochs are allowed to take them.

In [ ]:
RANKS = [4, 8, 16, 32]
ALPHA_TO_RANK_RATIO = 2

sweep_results = []
sweep_histories = {}

for r in RANKS:
    print(f"\n{'=' * 50}\nLoRA r={r}\n{'=' * 50}")

    base_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=num_labels
    )
    model = get_peft_model(
        base_model,
        LoraConfig(
            task_type=TaskType.SEQ_CLS,
            r=r,
            lora_alpha=r * ALPHA_TO_RANK_RATIO,
            lora_dropout=0.1,
            target_modules=["query", "value"],
        ),
    ).to(device)

    trainable, total = count_trainable_params(model)

    args = TrainingArguments(
        output_dir=f"./lora_sweep_r{r}",
        learning_rate=2e-4,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=64,
        num_train_epochs=20,
        warmup_ratio=0.06,          # a fresh 77-way head at this LR diverges without it
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        logging_steps=100,
        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
    )

    reset_peak_memory()
    start = time.time()
    trainer.train()
    train_time = time.time() - start
    peak_mem = peak_memory_mb()

    sweep_histories[str(r)] = epoch_history(trainer)
    test_metrics = trainer.evaluate(test_ds)

    sweep_results.append({
        "rank": r,
        "alpha": r * ALPHA_TO_RANK_RATIO,
        "trainable_params": trainable,
        "trainable_pct": round(100 * trainable / total, 3),
        "accuracy": test_metrics["eval_accuracy"],
        "macro_f1": test_metrics["eval_macro_f1"],
        "train_time_s": round(train_time, 1),
        "peak_mem_mb": round(peak_mem, 1) if peak_mem else None,
        "epochs_run": round(trainer.state.epoch, 1),
        "steps_run": trainer.state.global_step,
    })

    print(f"r={r}: test acc={test_metrics['eval_accuracy']:.4f}  "
          f"macro F1={test_metrics['eval_macro_f1']:.4f}  "
          f"trainable={trainable:,} ({100 * trainable / total:.2f}%)  "
          f"epochs={trainer.state.epoch:.0f}  time={train_time:.1f}s")

    del model, base_model, trainer
    release_cuda()

sweep_df = pd.DataFrame(sweep_results)
sweep_df

## 4. Where returns stop

Each doubling of `r` doubles the adapter parameters. The question is what the last doubling bought —
the cell below reports the accuracy delta per step of rank so the point of diminishing returns is a
number rather than an impression of the curve.

In [ ]:
deltas = sweep_df.copy()
deltas["acc_gain_pts"] = (deltas["accuracy"].diff() * 100).round(2)
deltas["params_added"] = deltas["trainable_params"].diff()
deltas["pts_per_100k_params"] = (
    deltas["acc_gain_pts"] / (deltas["params_added"] / 100_000)
).round(3)

print(deltas[["rank", "accuracy", "acc_gain_pts", "params_added", "pts_per_100k_params"]]
      .to_string(index=False))

best = sweep_df.loc[sweep_df["accuracy"].idxmax()]
print(f"\nBest rank: r={int(best['rank'])} at {best['accuracy']:.4f} test accuracy "
      f"({int(best['trainable_params']):,} trainable params)")
if FULL_FINETUNE_ACCURACY:
    print(f"Full fine-tune reference: {FULL_FINETUNE_ACCURACY:.4f} "
          f"({100 * best['accuracy'] / FULL_FINETUNE_ACCURACY:.1f}% retained at best rank)")

print("\n--- paste into README ---\n")
print("| Rank | Alpha | Trainable params | Accuracy | Macro F1 | Epochs | Train time (s) |")
print("|---|---|---|---|---|---|---|")
for _, r in sweep_df.iterrows():
    print(f"| {int(r['rank'])} | {int(r['alpha'])} | {int(r['trainable_params']):,} | "
          f"{r['accuracy']:.4f} | {r['macro_f1']:.4f} | {r['epochs_run']:.0f} | "
          f"{r['train_time_s']} |")

## 5. Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(sweep_df["rank"], sweep_df["accuracy"], marker="o", color="#eb6834", label="LoRA")
if FULL_FINETUNE_ACCURACY is not None:
    axes[0].axhline(FULL_FINETUNE_ACCURACY, color="#2a78d6", linestyle="--",
                    label="Full fine-tune")
axes[0].set_xscale("log", base=2)
axes[0].set_xlabel("Rank (r)")
axes[0].set_ylabel("Test accuracy")
axes[0].set_title("Accuracy vs rank")
axes[0].legend()

axes[1].plot(sweep_df["rank"], sweep_df["trainable_params"], marker="o", color="#1baf7a")
axes[1].set_xscale("log", base=2)
axes[1].set_yscale("log")
axes[1].set_xlabel("Rank (r)")
axes[1].set_ylabel("Trainable parameters")
axes[1].set_title("Trainable parameters vs rank")

for r, hist in sweep_histories.items():
    axes[2].plot([h["epoch"] for h in hist], [h["accuracy"] for h in hist],
                 marker=".", label=f"r={r}")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Validation accuracy")
axes[2].set_title("Convergence by rank")
axes[2].legend()

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/02_rank_sweep.png", dpi=150)
plt.show()

In [ ]:
sweep_df.to_csv(f"{RESULTS_DIR}/02_rank_sweep.csv", index=False)
with open(f"{RESULTS_DIR}/02_rank_sweep.json", "w") as fh:
    json.dump(sweep_df.to_dict(orient="records"), fh, indent=2)
with open(f"{RESULTS_DIR}/02_rank_sweep_curves.json", "w") as fh:
    json.dump(sweep_histories, fh, indent=2)

print("Wrote:")
for name in ["02_rank_sweep.csv", "02_rank_sweep.json",
             "02_rank_sweep_curves.json", "02_rank_sweep.png"]:
    print(" ", os.path.join(RESULTS_DIR, name))